# GL-01 Message Passing

Audience:
- Learners beginning Module 13 on graph learning.

Learning goals:
- Build adjacency, degree, and Laplacian matrices for a small graph.
- Compare mean aggregation with normalized graph propagation.
- Verify permutation equivariance numerically.
- Connect one-step aggregation to Laplacian smoothing.


In [ ]:
from __future__ import annotations

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from notebook path.")


REPO_ROOT = find_repo_root(Path.cwd())
MODULE_SRC = REPO_ROOT / "modules" / "13-graph-learning" / "src"
if str(MODULE_SRC) not in sys.path:
    sys.path.insert(0, str(MODULE_SRC))

from graph_learning_utils import build_adjacency, degree_matrix, normalized_adjacency, normalized_laplacian, row_mean_aggregate


## Step 1 - Define a toy graph and node features

The graph has five nodes and six undirected edges. Each node starts with a two-dimensional feature vector.


In [ ]:
edges = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 4), (3, 4)]
adjacency = build_adjacency(num_nodes=5, edges=edges)
degrees = degree_matrix(adjacency)
laplacian = degrees - adjacency
node_features = np.array(
    [
        [1.0, 0.0],
        [0.0, 1.0],
        [1.0, 1.0],
        [0.2, 1.4],
        [1.2, 0.1],
    ]
)

{"adjacency": adjacency, "degree_matrix": degrees, "laplacian": laplacian, "features": node_features}


## Step 2 - Compare two one-hop propagation rules

A mean aggregator averages neighbor features. A normalized propagation operator rescales by inverse square root degree, which is the GCN-style choice.


In [ ]:
mean_messages = row_mean_aggregate(node_features, adjacency, include_self=True)
gcn_operator = normalized_adjacency(adjacency, add_self_loops=True)
gcn_messages = gcn_operator @ node_features

{
    "mean_aggregation": np.round(mean_messages, 3),
    "gcn_style_propagation": np.round(gcn_messages, 3),
}


## Step 3 - Verify permutation equivariance

Relabeling nodes should only relabel the outputs. The numerical check below permutes the graph, reruns aggregation, and then maps the result back.


In [ ]:
permutation = np.array([2, 0, 4, 1, 3])
inverse_permutation = np.argsort(permutation)
P = np.eye(len(permutation))[permutation]

adjacency_permuted = P @ adjacency @ P.T
features_permuted = P @ node_features

original_output = row_mean_aggregate(node_features, adjacency, include_self=True)
permuted_output = row_mean_aggregate(features_permuted, adjacency_permuted, include_self=True)
mapped_back = P.T @ permuted_output

{
    "max_absolute_difference": float(np.abs(original_output - mapped_back).max()),
    "equivariant": bool(np.allclose(original_output, mapped_back)),
}


## Step 4 - View aggregation as smoothing

The normalized Laplacian measures how rapidly a feature varies over edges. After one propagation step, the quadratic form usually drops because the signal becomes smoother on the graph.


In [ ]:
signal = node_features[:, 0]
normalized_lap = normalized_laplacian(adjacency)
smoothed_signal = gcn_operator @ signal

roughness_before = float(signal.T @ normalized_lap @ signal)
roughness_after = float(smoothed_signal.T @ normalized_lap @ smoothed_signal)

fig, ax = plt.subplots(figsize=(6.8, 3.8))
positions = np.arange(len(signal))
ax.plot(positions, signal, marker="o", label="original feature")
ax.plot(positions, smoothed_signal, marker="o", label="after normalized propagation")
ax.set_title("One message-passing step smooths the graph signal")
ax.set_xlabel("node index")
ax.set_ylabel("feature value")
ax.legend()
fig.tight_layout()
plt.close(fig)

{"roughness_before": roughness_before, "roughness_after": roughness_after}


## Interpretation

This notebook shows the core mechanics behind graph learning:
- graph layers are built from adjacency-aware aggregation;
- the layer output should be equivariant to node relabeling; and
- normalized propagation behaves like a smoothing operator tied to the Laplacian.

Suggested extensions:
- replace mean aggregation with max aggregation;
- add edge weights and compare the propagated features; and
- apply two or three propagation steps and observe when node features begin to collapse together.
